# GPT-2 preentrenado como traductor español → inglés

Mismo problema, misma data y mismas métricas que el notebook del GPT propio, pero partiendo de los pesos preentrenados de **GPT-2 small (124M)** en lugar de pesos aleatorios. La arquitectura es la misma clase `GPT` estilo minGPT del notebook anterior: GPT-2 tiene exactamente esos bloques (LayerNorm → atención causal → LayerNorm → MLP con GELU), así que se cargan sus pesos en esa clase, igual que hace `GPT.from_pretrained` en minGPT.

Al final se comparan ambos modelos sobre el mismo test.

**Configuración de la sesión de Kaggle**
- Accelerator: **GPU T4 x2**
- Internet: **activado** (OPUS-100, pesos y tokenizador de GPT-2 desde Hugging Face)
- Input: el zip de **spa-eng** subido como dataset
- Input opcional: el output del notebook anterior (`best_model.pt` y `tokenizer.json`). Si está, se usa su tokenizador y se ponen las traducciones de ambos modelos lado a lado.

**Flujo:** 1. Setup → 2. Datos → 3. Tokenizadores → 4. Dataset → 5. GPT-2 preentrenado → 6. Trainer → 7. ¿1 o 2 GPUs? → 8. Fine-tuning → 9. Métricas → 10. Testing → 11. Comparación

## 1. Setup

In [ ]:
import importlib.util
import subprocess
import sys

for package in ("datasets", "sacrebleu"):
    if importlib.util.find_spec(package) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=True)

In [ ]:
import copy
import csv
import math
import time
from dataclasses import asdict, dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sacrebleu
import torch
import torch.nn as nn
from datasets import load_dataset
from matplotlib.ticker import MaxNLocator
from tokenizers import Tokenizer, decoders, models, pre_tokenizers, trainers
from torch.nn import functional as F
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, GPT2LMHeadModel

SEED = 1337
DATA_PATH = next(Path("/kaggle/input").rglob("spa.txt"))
WORK_DIR = Path("/kaggle/working")
GPT2_NAME = "distilbert/distilgpt2"
OWN_DIR = next((p.parent for p in Path("/kaggle/input").rglob("best_model.pt")), None)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
N_GPUS = torch.cuda.device_count()

PALETTE = ["#2a78d6", "#eb6834", "#1baf7a"]
plt.rcParams.update({
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.prop_cycle": plt.cycler(color=PALETTE),
    "lines.linewidth": 2,
})
pd.set_option("display.max_colwidth", 100)

print("dataset:", DATA_PATH)
print("modelo propio:", OWN_DIR or "no adjuntado")
print("torch:", torch.__version__, "| GPUs:", N_GPUS)
for i in range(N_GPUS):
    props = torch.cuda.get_device_properties(i)
    print(f"  cuda:{i} {props.name} | {props.total_memory / 1e9:.1f} GB")

## 2. Datos

Es el mismo pipeline del notebook anterior (el análisis completo está ahí): spa-eng completo, OPUS-100 limpio con los mismos filtros y la misma división con la misma semilla. Así train, validación y test contienen exactamente los mismos pares.

In [ ]:
raw = pd.read_csv(DATA_PATH, sep="\t", names=["en", "es"], quoting=csv.QUOTE_NONE)
print(f"{len(raw):,} pares")
raw.sample(8, random_state=SEED)

In [ ]:
opus = load_dataset("Helsinki-NLP/opus-100", "en-es")
opus_raw = {
    split: opus[split].flatten().to_pandas().rename(columns={"translation.en": "en", "translation.es": "es"})[["en", "es"]]
    for split in ("train", "validation", "test")
}
print(pd.Series({split: len(frame) for split, frame in opus_raw.items()}, name="pares").to_string())
opus_raw["train"].sample(10, random_state=SEED)

In [ ]:
class OpusCleaner:
    letters = r"[A-Za-zÁÉÍÓÚÜÑáéíóúüñ]"
    markup = r"https?://|www\.|&\s*\w+;"
    es_words = {"el", "la", "los", "las", "que", "de", "del", "y", "en", "una", "por", "para", "con", "es", "se"}
    en_words = {"the", "and", "you", "is", "are", "this", "that", "with", "what", "of", "to", "it"}

    def __init__(self, min_ratio=0.5, max_ratio=2.0, max_words=40, min_letter_share=0.5):
        self.min_ratio = min_ratio
        self.max_ratio = max_ratio
        self.max_words = max_words
        self.min_letter_share = min_letter_share

    @staticmethod
    def normalize(series):
        return series.str.replace(r"^\s*-\s*", "", regex=True).str.replace(r"\s+", " ", regex=True).str.strip()

    def letter_share(self, series):
        return series.str.count(self.letters) / series.str.count(r"\S").clip(lower=1)

    @staticmethod
    def count_words(series, vocabulary):
        return series.str.findall(r"[a-záéíóúüñ']+").map(lambda words: sum(w in vocabulary for w in words))

    def audit(self, frame):
        es, en = self.normalize(frame["es"]), self.normalize(frame["en"])
        es_lower, en_lower = es.str.lower(), en.str.lower()
        ratio = es.str.len() / en.str.len().clip(lower=1)
        checks = {
            "vacío": (es == "") | (en == ""),
            "sin traducir": es_lower == en_lower,
            "url o markup": es.str.contains(self.markup) | en.str.contains(self.markup),
            "letras de canciones": es.str.contains("[♪¶#]") | en.str.contains("[♪¶#]"),
            "pocas letras": (self.letter_share(es) < self.min_letter_share) | (self.letter_share(en) < self.min_letter_share),
            "demasiado larga": (es.str.split().str.len() > self.max_words) | (en.str.split().str.len() > self.max_words),
            "longitud incoherente": ~ratio.between(self.min_ratio, self.max_ratio),
            "idioma cruzado": (self.count_words(es_lower, self.en_words) > self.count_words(es_lower, self.es_words))
                              | (self.count_words(en_lower, self.es_words) > self.count_words(en_lower, self.en_words)),
        }
        reason = pd.Series(np.nan, index=frame.index, dtype=object)
        for name, mask in reversed(list(checks.items())):
            reason[mask] = name
        return pd.DataFrame({"es": es, "en": en, "motivo": reason})

    def clean(self, frame):
        audited = self.audit(frame)
        valid = audited[audited["motivo"].isna()]
        clean = valid[["es", "en"]].drop_duplicates().reset_index(drop=True)
        summary = audited["motivo"].value_counts()
        summary["duplicado"] = len(valid) - len(clean)
        summary["conservado"] = len(clean)
        return clean, audited, summary

cleaner = OpusCleaner()
opus_clean, opus_audit, opus_summary = {}, {}, {}
for split, frame in opus_raw.items():
    opus_clean[split], opus_audit[split], opus_summary[split] = cleaner.clean(frame)

summary = pd.DataFrame(opus_summary).fillna(0).astype(int)
summary["% train"] = (100 * summary["train"] / len(opus_raw["train"])).round(2)
summary

In [ ]:
def split_by_source(frame, val_frac, test_frac, seed):
    sources = frame["es"].drop_duplicates().sample(frac=1, random_state=seed).to_numpy()
    n_val, n_test = int(len(sources) * val_frac), int(len(sources) * test_frac)
    groups = {"test": sources[:n_test], "val": sources[n_test:n_test + n_val], "train": sources[n_test + n_val:]}
    return {name: frame[frame["es"].isin(set(group))].reset_index(drop=True) for name, group in groups.items()}


OPUS_TRAIN_PAIRS = 400_000

tatoeba = split_by_source(raw, val_frac=0.05, test_frac=0.05, seed=SEED)
opus_parts = {
    "train": opus_clean["train"].sample(min(OPUS_TRAIN_PAIRS, len(opus_clean["train"])), random_state=SEED),
    "val": opus_clean["validation"],
    "test": opus_clean["test"],
}
splits = {
    name: pd.concat([tatoeba[name].assign(source="spa-eng"), opus_parts[name].assign(source="opus")], ignore_index=True)
    for name in ("train", "val", "test")
}
held_out = set(splits["val"]["es"]) | set(splits["test"]["es"])
splits["train"] = splits["train"][~splits["train"]["es"].isin(held_out)].drop_duplicates(["es", "en"]).reset_index(drop=True)

pd.DataFrame({name: frame["source"].value_counts() for name, frame in splits.items()}).T.assign(total=lambda d: d.sum(axis=1))

## 3. Tokenizadores

- **GPT-2** obliga a usar su propio tokenizador: sus pesos están atados a su vocabulario de 50.257 tokens, entrenado sobre todo con texto en inglés. No tiene tokens `<es>` ni `<en>`, así que el formato se escribe como texto: `<|endoftext|>Spanish: ... \nEnglish: ...<|endoftext|>`.
- **El BPE propio de 16k** se usa para aplicar exactamente el mismo filtro de longitud que el notebook anterior. Si adjuntaste su output se carga su `tokenizer.json`; si no, se vuelve a entrenar con la misma data, que da el mismo vocabulario porque el entrenamiento del BPE es determinista.

In [ ]:
class BilingualTokenizer:
    specials = ["<pad>", "<es>", "<en>", "<eos>"]

    def __init__(self, tokenizer):
        self.tokenizer = tokenizer
        self.pad_id, self.es_id, self.en_id, self.eos_id = (tokenizer.token_to_id(t) for t in self.specials)

    @classmethod
    def train(cls, texts, vocab_size):
        tokenizer = Tokenizer(models.BPE())
        tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
        tokenizer.decoder = decoders.ByteLevel()
        trainer = trainers.BpeTrainer(vocab_size=vocab_size, special_tokens=cls.specials, show_progress=False,
                                      initial_alphabet=pre_tokenizers.ByteLevel.alphabet())
        tokenizer.train_from_iterator(texts, trainer)
        return cls(tokenizer)

    @property
    def vocab_size(self):
        return self.tokenizer.get_vocab_size()

    def encode(self, texts):
        return [e.ids for e in self.tokenizer.encode_batch(list(texts))]

    def decode(self, ids):
        return self.tokenizer.decode(ids)

    def pieces(self, ids):
        return [self.tokenizer.decode([i], skip_special_tokens=False) for i in ids]

    def prompt(self, source_ids):
        return [self.es_id, *source_ids, self.en_id]

class GPT2TranslationTokenizer:
    def __init__(self, name):
        self.tokenizer = AutoTokenizer.from_pretrained(name)
        self.eos_id = self.pad_id = self.tokenizer.eos_token_id
        self.es_prefix = self.tokenizer("Spanish:", add_special_tokens=False)["input_ids"]
        self.en_prefix = self.tokenizer("\nEnglish:", add_special_tokens=False)["input_ids"]

    @property
    def vocab_size(self):
        return len(self.tokenizer)

    def encode(self, texts):
        return self.tokenizer([" " + t for t in texts], add_special_tokens=False)["input_ids"]

    def decode(self, ids):
        return self.tokenizer.decode(ids, skip_special_tokens=True).strip()

    def pieces(self, ids):
        return [self.tokenizer.decode([i]) for i in ids]

    def prompt(self, source_ids):
        return [self.eos_id, *self.es_prefix, *source_ids, *self.en_prefix]

In [ ]:
if OWN_DIR:
    own_tokenizer = BilingualTokenizer(Tokenizer.from_file(str(OWN_DIR / "tokenizer.json")))
else:
    own_tokenizer = BilingualTokenizer.train(pd.concat([splits["train"]["es"], splits["train"]["en"]]), vocab_size=16000)
tokenizer = GPT2TranslationTokenizer(GPT2_NAME)
print(f"vocabulario propio: {own_tokenizer.vocab_size:,} | vocabulario GPT-2: {tokenizer.vocab_size:,}")

for sentence in ["¿Dónde está la estación de tren?", "Where is the train station?"]:
    print(f"\n{sentence}")
    print("  propio:", " | ".join(own_tokenizer.pieces(own_tokenizer.encode([sentence])[0])))
    print("  GPT-2: ", " | ".join(tokenizer.pieces(tokenizer.encode([sentence])[0])))

In [ ]:
sample = splits["train"].sample(5000, random_state=SEED)
token_counts = pd.DataFrame({
    name: {lang: np.mean([len(ids) for ids in tok.encode(sample[lang])]) for lang in ("es", "en")}
    for name, tok in [("BPE propio (16k)", own_tokenizer), ("GPT-2 (50k)", tokenizer)]
}).rename(index={"es": "español", "en": "inglés"})

fig, ax = plt.subplots(figsize=(8, 4))
width = 0.35
positions = np.arange(len(token_counts.index))
for offset, (color, name) in zip((-width / 2, width / 2), zip(PALETTE, token_counts.columns)):
    ax.bar(positions + offset, token_counts[name], width=width, color=color, label=name)
ax.set_xticks(positions, token_counts.index)
ax.set(title="Tokens promedio por frase", ylabel="tokens")
ax.legend()
plt.show()
token_counts.round(2)

Mismo filtro de longitud que el notebook anterior, aplicado con el BPE propio:

In [ ]:
BLOCK_SIZE = 96

for name in splits:
    own_tokens = np.array([len(s) + len(t) + 3 for s, t in zip(own_tokenizer.encode(splits[name]["es"]), own_tokenizer.encode(splits[name]["en"]))])
    keep = own_tokens <= BLOCK_SIZE + 1
    splits[name] = splits[name][keep].reset_index(drop=True)
    print(f"{name:5s}: {len(splits[name]):,} pares ({keep.mean():.2%} conservados)")

gpt2_tokens = np.array([len(tokenizer.prompt(s)) + len(t) + 1 for s, t in zip(tokenizer.encode(splits["train"]["es"]), tokenizer.encode(splits["train"]["en"]))])
print(f"\ntokens GPT-2 por par en train: media {gpt2_tokens.mean():.1f} | p99 {np.percentile(gpt2_tokens, 99):.0f} | máx {gpt2_tokens.max()} (contexto de GPT-2: 1024)")

## 4. Dataset

La misma clase `TranslationDataset` del notebook anterior, ahora con el tokenizador de GPT-2. La loss se sigue calculando solo sobre la traducción al inglés y el `<|endoftext|>` final.

In [ ]:
class TranslationDataset(Dataset):
    def __init__(self, frame, tokenizer):
        self.tokenizer = tokenizer
        self.sources = tokenizer.encode(frame["es"])
        self.targets = tokenizer.encode(frame["en"])

    def __len__(self):
        return len(self.sources)

    def __getitem__(self, i):
        prompt = self.tokenizer.prompt(self.sources[i])
        sequence = torch.tensor(prompt + self.targets[i] + [self.tokenizer.eos_id])
        x, y = sequence[:-1], sequence[1:].clone()
        y[:len(prompt) - 1] = -1
        return x, y

    def collate(self, batch):
        xs, ys = zip(*batch)
        x = nn.utils.rnn.pad_sequence(xs, batch_first=True, padding_value=self.tokenizer.pad_id)
        y = nn.utils.rnn.pad_sequence(ys, batch_first=True, padding_value=-1)
        return x, y


datasets = {name: TranslationDataset(frame, tokenizer) for name, frame in splits.items()}

x, y = datasets["train"][0]
print("x:", " | ".join(tokenizer.pieces(x.tolist())))
print("y:", " | ".join("·" if t == -1 else tokenizer.pieces([t])[0] for t in y.tolist()))

## 5. GPT-2 preentrenado

Es la clase `GPT` del notebook anterior con un único método nuevo, `from_pretrained`, que copia los pesos de GPT-2 desde Hugging Face:
- GPT-2 usa capas `Conv1D` que guardan sus pesos transpuestos respecto a `nn.Linear`, así que las matrices de los bloques se transponen al copiarlas.
- GPT-2 comparte los pesos entre `wte` y `lm_head`, así que aquí también se atan.

In [ ]:
@dataclass
class GPTConfig:
    vocab_size: int
    block_size: int
    n_layer: int = 6
    n_head: int = 8
    n_embd: int = 512
    dropout: float = 0.1

class CausalSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd)
        self.c_proj = nn.Linear(config.n_embd, config.n_embd)
        self.attn_dropout = nn.Dropout(config.dropout)
        self.resid_dropout = nn.Dropout(config.dropout)
        self.n_head = config.n_head
        self.store_attention = False
        self.attention = None

    def forward(self, x):
        B, T, C = x.size()
        q, k, v = self.c_attn(x).split(C, dim=2)
        q, k, v = (t.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) for t in (q, k, v))
        if self.store_attention:
            att = (q @ k.transpose(-2, -1)) / math.sqrt(k.size(-1))
            att = att.masked_fill(~torch.ones(T, T, dtype=torch.bool, device=x.device).tril(), float("-inf")).softmax(dim=-1)
            self.attention = att.detach()
            y = self.attn_dropout(att) @ v
        else:
            y = F.scaled_dot_product_attention(q, k, v, dropout_p=self.attn_dropout.p if self.training else 0.0, is_causal=True)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.c_proj(y))

class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.ln_1 = nn.LayerNorm(config.n_embd)
        self.attn = CausalSelfAttention(config)
        self.ln_2 = nn.LayerNorm(config.n_embd)
        self.mlp = nn.Sequential(
            nn.Linear(config.n_embd, 4 * config.n_embd),
            nn.GELU(approximate="tanh"),
            nn.Linear(4 * config.n_embd, config.n_embd),
            nn.Dropout(config.dropout),
        )

    def forward(self, x):
        x = x + self.attn(self.ln_1(x))
        return x + self.mlp(self.ln_2(x))


class GPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.transformer = nn.ModuleDict(dict(
            wte=nn.Embedding(config.vocab_size, config.n_embd),
            wpe=nn.Embedding(config.block_size, config.n_embd),
            drop=nn.Dropout(config.dropout),
            h=nn.ModuleList([Block(config) for _ in range(config.n_layer)]),
            ln_f=nn.LayerNorm(config.n_embd),
        ))
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        self.apply(self._init_weights)
        for name, p in self.named_parameters():
            if name.endswith("c_proj.weight") or name.endswith("mlp.2.weight"):
                nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * config.n_layer))

    @staticmethod
    def _init_weights(module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
        if isinstance(module, nn.Linear) and module.bias is not None:
            nn.init.zeros_(module.bias)

    @classmethod
    def from_pretrained(cls, name, dropout=0.1):
        hf_model = GPT2LMHeadModel.from_pretrained(name)
        c = hf_model.config
        model = cls(GPTConfig(vocab_size=c.vocab_size, block_size=c.n_positions, n_layer=c.n_layer, n_head=c.n_head, n_embd=c.n_embd, dropout=dropout))
        model.lm_head.weight = model.transformer.wte.weight
        hf_state = hf_model.state_dict()
        transpose = type(hf_model.transformer.h[0].attn.c_attn).__name__ == "Conv1D"
        with torch.no_grad():
            for param_name, p in model.named_parameters():
                hf_name = param_name.replace(".mlp.0.", ".mlp.c_fc.").replace(".mlp.2.", ".mlp.c_proj.")
                weight = hf_state[hf_name]
                p.copy_(weight.t() if transpose and weight.dim() == 2 and ".h." in hf_name else weight)
        return model

    def forward(self, idx, targets=None):
        pos = torch.arange(idx.size(1), device=idx.device)
        x = self.transformer.drop(self.transformer.wte(idx) + self.transformer.wpe(pos))
        for block in self.transformer.h:
            x = block(x)
        logits = self.lm_head(self.transformer.ln_f(x))
        if targets is None:
            return logits
        return F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1), ignore_index=-1).unsqueeze(0)

    def configure_optimizers(self, config):
        decay = [p for n, p in self.named_parameters() if p.dim() >= 2 and "wte" not in n and "wpe" not in n]
        no_decay = [p for n, p in self.named_parameters() if p.dim() < 2 or "wte" in n or "wpe" in n]
        groups = [{"params": decay, "weight_decay": config.weight_decay}, {"params": no_decay, "weight_decay": 0.0}]
        return torch.optim.AdamW(groups, lr=config.learning_rate, betas=config.betas)

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, eos_id):
        finished = torch.zeros(idx.size(0), dtype=torch.bool, device=idx.device)
        for _ in range(max_new_tokens):
            logits = self(idx[:, -self.config.block_size:])
            next_id = logits[:, -1, :].argmax(dim=-1, keepdim=True)
            next_id = next_id.masked_fill(finished[:, None], eos_id)
            finished |= next_id.squeeze(1) == eos_id
            idx = torch.cat((idx, next_id), dim=1)
            if finished.all():
                break
        return idx

    def parameter_table(self):
        rows = []
        for name, p in self.named_parameters():
            parts = name.split(".")
            if parts[0] == "lm_head":
                layer, group = "lm_head", "lm_head"
            elif parts[1] != "h":
                layer, group = parts[1], {"wte": "token_embedding", "wpe": "position_embedding", "ln_f": "layernorm"}[parts[1]]
            else:
                layer, group = f"block_{parts[2]}", {"attn": "attention", "mlp": "mlp"}.get(parts[3], "layernorm")
            rows.append(dict(name=name, layer=layer, group=group, shape=tuple(p.shape), params=p.numel()))
        return pd.DataFrame(rows)

In [ ]:
pretrained = GPT.from_pretrained(GPT2_NAME).eval()
hf_model = GPT2LMHeadModel.from_pretrained(GPT2_NAME).eval()

check_ids = torch.tensor([tokenizer.tokenizer("The quick brown fox jumps over the lazy dog.", add_special_tokens=False)["input_ids"]])
with torch.no_grad():
    max_diff = (pretrained(check_ids) - hf_model(check_ids).logits).abs().max().item()
print(f"diferencia máxima de logits frente al GPT-2 de Hugging Face: {max_diff:.2e}")
del hf_model

print(pretrained)
pd.Series(asdict(pretrained.config), name="valor", dtype=object).to_frame()

In [ ]:
params = pretrained.parameter_table()
n_params = params["params"].sum()
print(f"Parámetros totales: {n_params / 1e6:.2f}M (lm_head comparte pesos con token_embedding)")

by_group = params.groupby("group")["params"].sum().sort_values()
by_layer = params.groupby("layer", sort=False)["params"].sum()

fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
axes[0].barh(by_group.index, by_group.values / 1e6, color=PALETTE[0])
axes[0].set(title="Parámetros por componente", xlabel="millones")
axes[1].bar(by_layer.index, by_layer.values / 1e6, color=PALETTE[0])
axes[1].set(title="Parámetros por capa", ylabel="millones")
axes[1].tick_params(axis="x", rotation=45)
fig.tight_layout()
plt.show()

by_group.to_frame("parámetros").assign(**{"%": lambda d: (100 * d["parámetros"] / n_params).round(2)})[::-1]

**Antes del fine-tuning.** GPT-2 ya sabe continuar texto en inglés, pero no sabe traducir: ante el formato `Spanish: ... English:` solo continúa el texto como puede.

In [ ]:
pretrained = pretrained.to(DEVICE)
prompt_ids = tokenizer.tokenizer("The capital of France is", add_special_tokens=False)["input_ids"]
continuation = pretrained.generate(torch.tensor([[tokenizer.eos_id, *prompt_ids]], device=DEVICE), 12, tokenizer.eos_id)[0, len(prompt_ids) + 1:].tolist()
print("The capital of France is" + tokenizer.tokenizer.decode(continuation))

before_sentences = ["¿Me puedes ayudar a encontrar mis llaves?", "El perro de mi vecino ladra toda la noche."]
before = []
for sentence in before_sentences:
    prompt = tokenizer.prompt(tokenizer.encode([sentence])[0])
    output = pretrained.generate(torch.tensor([prompt], device=DEVICE), 20, tokenizer.eos_id)[0, len(prompt):].tolist()
    before.append(tokenizer.decode(output))
del pretrained
torch.cuda.empty_cache()
pd.DataFrame({"español": before_sentences, "GPT-2 sin fine-tuning": before})

## 6. Trainer

Es el mismo `Trainer` del notebook anterior. Solo cambian los valores de `TrainConfig`, porque ajustar un modelo preentrenado no es lo mismo que entrenar desde cero:
- **learning rate de 1e-4** en lugar de 6e-4, para no destruir lo que GPT-2 ya sabe;
- **pocas épocas**: el modelo parte de un buen punto y cada época cuesta bastante más, porque GPT-2 es 3.5 veces más grande y su tokenizador parte el español en más tokens;
- **batch de 64**, para que los logits sobre 50.257 tokens quepan en la memoria de las T4.

In [ ]:
@dataclass
class TrainConfig:
    batch_size: int = 64
    learning_rate: float = 1e-3
    min_lr_ratio: float = 0.1
    warmup_steps: int = 500
    max_epochs: int = 5
    weight_decay: float = 0.1
    betas: tuple = (0.9, 0.95)
    grad_norm_clip: float = 1.0
    patience: int = 2
    log_interval: int = 50

class Trainer:
    def __init__(self, model, config, train_ds, val_ds, n_gpus):
        self.model = model.to(DEVICE)
        self.net = nn.DataParallel(model) if n_gpus > 1 else model
        self.config = config
        self.optimizer = model.configure_optimizers(config)
        self.use_amp = DEVICE.type == "cuda"
        self.scaler = torch.amp.GradScaler("cuda", enabled=self.use_amp)
        loader_args = dict(batch_size=config.batch_size, num_workers=2, pin_memory=self.use_amp)
        self.train_loader = DataLoader(train_ds, shuffle=True, drop_last=True, collate_fn=train_ds.collate, **loader_args)
        self.val_loader = DataLoader(val_ds, collate_fn=val_ds.collate, **loader_args)
        self.total_steps = len(self.train_loader) * config.max_epochs
        self.step = 0
        self.history = {"steps": [], "epochs": []}

    def lr_at(self, step):
        c = self.config
        if step < c.warmup_steps:
            return c.learning_rate * (step + 1) / c.warmup_steps
        progress = min(1.0, (step - c.warmup_steps) / (self.total_steps - c.warmup_steps))
        return c.learning_rate * (c.min_lr_ratio + (1 - c.min_lr_ratio) * 0.5 * (1 + math.cos(math.pi * progress)))

    def train_step(self, x, y):
        lr = self.lr_at(self.step)
        for group in self.optimizer.param_groups:
            group["lr"] = lr
        x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
        with torch.amp.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=self.use_amp):
            loss = self.net(x, y).mean()
        self.optimizer.zero_grad(set_to_none=True)
        self.scaler.scale(loss).backward()
        self.scaler.unscale_(self.optimizer)
        grad_norm = nn.utils.clip_grad_norm_(self.model.parameters(), self.config.grad_norm_clip)
        self.scaler.step(self.optimizer)
        self.scaler.update()
        self.step += 1
        return loss.item(), grad_norm.item(), lr

    @torch.no_grad()
    def evaluate(self):
        self.net.eval()
        total, count = 0.0, 0
        for x, y in self.val_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            with torch.amp.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=self.use_amp):
                loss = self.net(x, y).mean()
            n = int((y != -1).sum())
            total, count = total + loss.item() * n, count + n
        self.net.train()
        return total / count

    def benchmark(self, steps, warmup=10):
        self.net.train()
        batches = iter(self.train_loader)
        tokens = 0
        for i in range(warmup + steps):
            if i == warmup:
                torch.cuda.synchronize()
                start, tokens = time.time(), 0
            x, y = next(batches)
            self.train_step(x, y)
            tokens += x.numel()
        torch.cuda.synchronize()
        return tokens / (time.time() - start)

    def fit(self, on_epoch_end=None):
        best_loss, best_state, bad_epochs = float("inf"), None, 0
        for epoch in range(1, self.config.max_epochs + 1):
            self.net.train()
            start, epoch_losses = time.time(), []
            for x, y in self.train_loader:
                loss, grad_norm, lr = self.train_step(x, y)
                epoch_losses.append(loss)
                if self.step % self.config.log_interval == 0:
                    self.history["steps"].append(dict(step=self.step, loss=loss, grad_norm=grad_norm, lr=lr))
            record = dict(epoch=epoch, step=self.step, train_loss=float(np.mean(epoch_losses)), val_loss=self.evaluate(),
                          time_sec=time.time() - start)
            record.update(on_epoch_end(self) if on_epoch_end else {})
            improved = record["val_loss"] < best_loss
            if improved:
                best_loss, bad_epochs = record["val_loss"], 0
                best_state = copy.deepcopy(self.model.state_dict())
                torch.save(best_state, WORK_DIR / "best_model.pt")
            else:
                bad_epochs += 1
            self.history["epochs"].append(record)
            extra = " | ".join(f"{k} {v:.2f}" for k, v in record.items() if k not in ("epoch", "step", "train_loss", "val_loss", "time_sec"))
            print(f"epoch {epoch:2d} | train {record['train_loss']:.4f} | val {record['val_loss']:.4f} | {extra} | "
                  f"{record['time_sec']:.0f}s{' | best' if improved else ''}")
            if bad_epochs >= self.config.patience:
                print(f"early stopping: {self.config.patience} épocas sin mejorar la val loss")
                break
        self.model.load_state_dict(best_state)
        return self.history


train_config = TrainConfig()
pd.Series(asdict(train_config), name="valor", dtype=object).to_frame()

## 7. ¿Conviene usar las 2 GPUs?

Mismo benchmark que antes, ahora con GPT-2. Con `nn.DataParallel` la GPU 0 tiene que copiar los 124M parámetros a la GPU 1 en cada paso, así que el resultado puede ser distinto al del modelo propio. También se estima cuánto tardará cada época.

In [ ]:
MIN_SPEEDUP = 1.2

def measure(n_gpus, steps=60):
    torch.manual_seed(SEED)
    trainer = Trainer(GPT.from_pretrained(GPT2_NAME), train_config, datasets["train"], datasets["val"], n_gpus)
    tokens_per_sec = trainer.benchmark(steps)
    del trainer
    torch.cuda.empty_cache()
    return tokens_per_sec


if N_GPUS >= 2:
    bench = pd.Series({f"{n} GPU": measure(n) for n in (1, 2)}, name="tokens/s")
    speedup = bench["2 GPU"] / bench["1 GPU"]
    USE_GPUS = 2 if speedup >= MIN_SPEEDUP else 1
    print(f"speedup 2 vs 1 GPU: {speedup:.2f}x -> se entrena con {USE_GPUS} GPU(s)")
    print(f"tiempo estimado por época: {gpt2_tokens.sum() / bench[f'{USE_GPUS} GPU'] / 60:.0f} min")

    fig, ax = plt.subplots(figsize=(6, 3.5))
    ax.bar(bench.index, bench.values, color=PALETTE[0], width=0.5)
    ax.set(title="Throughput de entrenamiento", ylabel="tokens/s")
    plt.show()
else:
    USE_GPUS = 1
    print("solo hay una GPU disponible -> se entrena con 1")

## 8. Fine-tuning

Igual que antes, al final de cada época se traduce la misma muestra de 1.000 frases de validación y se calcula su BLEU.

In [ ]:
class Translator:
    def __init__(self, model, tokenizer, batch_size=512):
        self.model = model
        self.tokenizer = tokenizer
        self.batch_size = batch_size

    @torch.no_grad()
    def translate(self, sentences, max_new_tokens=BLOCK_SIZE):
        self.model.eval()
        prompts = [self.tokenizer.prompt(ids) for ids in self.tokenizer.encode(sentences)]
        outputs = [None] * len(prompts)
        by_length = pd.Series(range(len(prompts))).groupby([len(p) for p in prompts])
        for length, rows in by_length:
            rows = rows.tolist()
            for start in range(0, len(rows), self.batch_size):
                chunk = rows[start:start + self.batch_size]
                idx = torch.tensor([prompts[i] for i in chunk], device=DEVICE)
                with torch.amp.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=DEVICE.type == "cuda"):
                    generated = self.model.generate(idx, max_new_tokens, self.tokenizer.eos_id)[:, length:].tolist()
                for i, ids in zip(chunk, generated):
                    outputs[i] = self.tokenizer.decode(ids[:ids.index(self.tokenizer.eos_id)] if self.tokenizer.eos_id in ids else ids)
        return outputs

    def score(self, frame):
        predictions = self.translate(frame["es"])
        return {"bleu": sacrebleu.corpus_bleu(predictions, [frame["en"].tolist()]).score,
                "chrf": sacrebleu.corpus_chrf(predictions, [frame["en"].tolist()]).score}


val_sample = splits["val"].sample(1000, random_state=SEED)

In [ ]:
torch.manual_seed(SEED)
model = GPT.from_pretrained(GPT2_NAME)
trainer = Trainer(model, train_config, datasets["train"], datasets["val"], USE_GPUS)
translator = Translator(model, tokenizer)

print(f"GPUs: {USE_GPUS} | steps por época: {len(trainer.train_loader)} | batch: {train_config.batch_size}")
start = time.time()
history = trainer.fit(on_epoch_end=lambda t: {"val_bleu": translator.score(val_sample)["bleu"]})
train_minutes = (time.time() - start) / 60
print(f"tiempo total: {train_minutes:.1f} min")

## 9. Métricas de entrenamiento

In [ ]:
steps = pd.DataFrame(history["steps"])
epochs = pd.DataFrame(history["epochs"]).assign(train_ppl=lambda d: np.exp(d["train_loss"]), val_ppl=lambda d: np.exp(d["val_loss"]))
epochs.round(4)

In [ ]:
best = epochs.loc[epochs["val_loss"].idxmin()]
fig, axes = plt.subplots(2, 2, figsize=(15, 9))

ax = axes[0, 0]
ax.plot(steps["step"], steps["loss"], color=PALETTE[0], alpha=0.3, linewidth=1)
ax.plot(steps["step"], steps["loss"].ewm(span=20).mean(), color=PALETTE[0], label="train (EMA)")
ax.plot(epochs["step"], epochs["val_loss"], color=PALETTE[1], marker="o", label="val")
ax.scatter([best["step"]], [best["val_loss"]], s=120, facecolors="none", edgecolors="black", zorder=3, label="mejor checkpoint")
ax.set(title="Loss (solo tokens en inglés)", xlabel="step", ylabel="cross-entropy")
ax.legend()

ax = axes[0, 1]
ax.plot(epochs["epoch"], epochs["val_bleu"], color=PALETTE[0], marker="o")
ax.set(title="BLEU en 1.000 frases de validación", xlabel="época", ylabel="BLEU")
ax.xaxis.set_major_locator(MaxNLocator(integer=True))

ax = axes[1, 0]
ax.plot(epochs["epoch"], epochs["train_ppl"], color=PALETTE[0], marker="o", label="train")
ax.plot(epochs["epoch"], epochs["val_ppl"], color=PALETTE[1], marker="o", label="val")
ax.set(title="Perplejidad por época", xlabel="época", ylabel="perplejidad")
ax.xaxis.set_major_locator(MaxNLocator(integer=True))
ax.legend()

ax = axes[1, 1]
ax.plot(steps["step"], steps["lr"], color=PALETTE[0])
ax.set(title="Learning rate (warmup + coseno)", xlabel="step", ylabel="lr")

fig.tight_layout()
plt.show()

print(f"mejor época: {int(best['epoch'])} | val loss {best['val_loss']:.4f} | val BLEU {best['val_bleu']:.2f}")

## 10. Testing

In [ ]:
del trainer
torch.cuda.empty_cache()
translator = Translator(model, tokenizer, batch_size=64)

In [ ]:
results = pd.DataFrame({name: translator.score(splits[name]) for name in ("val", "test")}).T
results.round(2)

### 10.1 Ejemplos del conjunto de test

In [ ]:
examples = splits["test"].sample(12, random_state=SEED).reset_index(drop=True)
examples["modelo"] = translator.translate(examples["es"])
examples.rename(columns={"es": "español", "en": "referencia"})[["español", "referencia", "modelo"]]

### 10.2 Calidad según la longitud de la frase

In [ ]:
test = splits["test"].assign(words=lambda d: d["es"].str.split().str.len())
test["modelo"] = translator.translate(test["es"])
test["bucket"] = pd.cut(test["words"], bins=[0, 3, 6, 9, 12, 100], labels=["1-3", "4-6", "7-9", "10-12", "13+"])

by_length = test.groupby("bucket", observed=True)[["modelo", "en"]].apply(
    lambda g: pd.Series({"frases": len(g), "bleu": sacrebleu.corpus_bleu(g["modelo"].tolist(), [g["en"].tolist()]).score})
)

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(by_length.index.astype(str), by_length["bleu"], color=PALETTE[0], width=0.6)
ax.set(title="BLEU en test según la longitud de la frase en español", xlabel="palabras", ylabel="BLEU")
plt.show()
by_length.round(2)

### 10.3 Prueba con tus propias frases

In [ ]:
my_sentences = [
    "Mañana voy a entrenar mi propio modelo de lenguaje.",
    "¿Me puedes ayudar a encontrar mis llaves?",
    "El perro de mi vecino ladra toda la noche.",
    "No tengo tiempo para ver películas esta semana.",
    "Ella siempre llega tarde a las reuniones del trabajo.",
    "Si llueve mañana, nos quedaremos en casa.",
    "Te amo Emily Guerrero.",
    "Barcelona es el mejor equipo del mundo."
]
pd.DataFrame({"español": my_sentences, "modelo": translator.translate(my_sentences)})

## 11. Comparación: GPT propio vs GPT-2 preentrenado

Los resultados del GPT propio son los de tu notebook anterior (misma data, mismo test, mejor checkpoint en la época 15).

In [ ]:
OWN_RESULTS = {
    "parámetros (M)": 35.35,
    "tokenizador": "BPE propio (16k)",
    "épocas entrenadas": 20,
    "mejor época": 15,
    "tiempo de entrenamiento (min)": 133.1,
    "val BLEU": 50.22,
    "val chrF": 65.94,
    "test BLEU": 50.82,
    "test chrF": 66.57,
}
OWN_VAL_BLEU = [30.40, 37.81, 40.56, 42.15, 44.51, 44.78, 45.81, 46.85, 46.62, 47.95,
                48.17, 48.64, 49.58, 50.40, 50.95, 50.34, 50.47, 50.35, 50.99, 51.01]
OWN_EPOCH_SECONDS = 395
OWN_BLEU_BY_LENGTH = {"1-3": 56.83, "4-6": 57.92, "7-9": 55.52, "10-12": 49.15, "13+": 35.42}

GPT2_RESULTS = {
    "parámetros (M)": round(sum(p.numel() for p in model.parameters()) / 1e6, 2),
    "tokenizador": "BPE de GPT-2 (50k)",
    "épocas entrenadas": len(epochs),
    "mejor época": int(best["epoch"]),
    "tiempo de entrenamiento (min)": round(train_minutes, 1),
    "val BLEU": round(results.loc["val", "bleu"], 2),
    "val chrF": round(results.loc["val", "chrf"], 2),
    "test BLEU": round(results.loc["test", "bleu"], 2),
    "test chrF": round(results.loc["test", "chrf"], 2),
}
pd.DataFrame({"GPT propio (desde cero)": OWN_RESULTS, "GPT-2 + fine-tuning": GPT2_RESULTS})

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 4.5))

ax = axes[0]
own_minutes = np.arange(1, len(OWN_VAL_BLEU) + 1) * OWN_EPOCH_SECONDS / 60
ax.plot(own_minutes, OWN_VAL_BLEU, marker="o", color=PALETTE[0], label="GPT propio (desde cero)")
ax.plot(epochs["time_sec"].cumsum() / 60, epochs["val_bleu"], marker="o", color=PALETTE[1], label="GPT-2 + fine-tuning")
ax.set(title="BLEU en validación según el tiempo de entrenamiento", xlabel="minutos de entrenamiento", ylabel="BLEU")
ax.legend()

ax = axes[1]
lengths = pd.DataFrame({"GPT propio (desde cero)": pd.Series(OWN_BLEU_BY_LENGTH), "GPT-2 + fine-tuning": by_length["bleu"].set_axis(by_length.index.astype(str))})
positions = np.arange(len(lengths))
width = 0.38
for offset, (color, name) in zip((-width / 2, width / 2), zip(PALETTE, lengths.columns)):
    ax.bar(positions + offset, lengths[name], width=width, color=color, label=name)
ax.set_xticks(positions, lengths.index)
ax.set(title="BLEU en test según la longitud de la frase", xlabel="palabras en español", ylabel="BLEU")
ax.legend()

fig.tight_layout()
plt.show()
lengths.round(2)

### 11.1 Traducciones lado a lado

Se necesita el output del notebook anterior como input (`best_model.pt` y `tokenizer.json`) para cargar el GPT propio.

In [ ]:
if OWN_DIR:
    own_model = GPT(GPTConfig(vocab_size=own_tokenizer.vocab_size, block_size=BLOCK_SIZE))
    own_model.load_state_dict(torch.load(OWN_DIR / "best_model.pt", map_location="cpu", weights_only=True))
    own_translator = Translator(own_model.to(DEVICE), own_tokenizer)

    side_by_side = pd.concat([
        examples[["es", "en"]].rename(columns={"es": "español", "en": "referencia"}),
        pd.DataFrame({"español": my_sentences, "referencia": ""}),
    ], ignore_index=True)
    side_by_side["GPT propio"] = own_translator.translate(side_by_side["español"])
    side_by_side["GPT-2"] = translator.translate(side_by_side["español"])
    display(side_by_side)
else:
    print("No se encontró best_model.pt en /kaggle/input: adjunta el output del notebook anterior para ver ambas traducciones.")